In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
print(os.listdir("/kaggle/input/datasets"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/amankumar2002"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/amankumar2002/ml4sci-test-1/dataset/train"))

In [ ]:
import os, random, warnings, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, auc, classification_report)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
TRAIN_DIR = "/kaggle/input/datasets/amankumar2002/ml4sci-test-1/dataset/train"
VAL_DIR   = "/kaggle/input/datasets/amankumar2002/ml4sci-test-1/dataset/val"

In [ ]:
print("\n" + "="*55)
print("  DATASET DIAGNOSTIC")
print("="*55)

def inspect_dataset(root):
    classes = sorted([c for c in os.listdir(root)
                      if os.path.isdir(os.path.join(root, c))])
    print(f"\nRoot: {root}")
    shapes, dtypes = [], []
    for cls in classes:
        files = [f for f in os.listdir(os.path.join(root, cls))
                 if f.endswith(".npy")]
        # sample one file
        sample_path = os.path.join(root, cls, files[0])
        arr = np.load(sample_path)
        shapes.append(arr.shape)
        dtypes.append(arr.dtype)
        print(f"  Class '{cls}': {len(files):>5} files  | "
              f"shape={arr.shape}  dtype={arr.dtype}  "
              f"min={arr.min():.3f}  max={arr.max():.3f}")
    return classes, shapes

CLASS_NAMES, shapes = inspect_dataset(TRAIN_DIR)
inspect_dataset(VAL_DIR)

# Determine channel count from shape
sample_shape = shapes[0]
if len(sample_shape) == 2:
    IN_CHANNELS = 1
    IMG_H, IMG_W = sample_shape
elif len(sample_shape) == 3:
    # Could be (C,H,W) or (H,W,C)
    if sample_shape[0] <= 4:       # likely (C,H,W)
        IN_CHANNELS   = sample_shape[0]
        IMG_H, IMG_W  = sample_shape[1], sample_shape[2]
    else:                           # likely (H,W,C)
        IMG_H, IMG_W  = sample_shape[0], sample_shape[1]
        IN_CHANNELS   = sample_shape[2]
else:
    raise ValueError(f"Unexpected shape: {sample_shape}")

NUM_CLASSES = len(CLASS_NAMES)
print(f"\nDetected: IN_CHANNELS={IN_CHANNELS}  IMG_H={IMG_H}  IMG_W={IMG_W}")
print(f"Classes : {CLASS_NAMES}")
print("="*55)

In [ ]:
class LensDataset(Dataset):
    """
    Robust .npy loader.
    • Handles (H,W), (C,H,W) and (H,W,C) layouts automatically.
    • Per-sample min-max clip before z-score to eliminate outlier pixels.
    • Augmentation: hflip / vflip / 90° rot  (train only).
    """
    def __init__(self, root_dir, mean=None, std=None, augment=False):
        self.mean    = mean
        self.std     = std
        self.augment = augment
        self.samples = []

        classes = sorted([c for c in os.listdir(root_dir)
                          if os.path.isdir(os.path.join(root_dir, c))])
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}

        for cls in classes:
            cls_path = os.path.join(root_dir, cls)
            for f in os.listdir(cls_path):
                if f.endswith(".npy"):
                    self.samples.append(
                        (os.path.join(cls_path, f), self.class_to_idx[cls])
                    )

    def __len__(self):
        return len(self.samples)

    def _load(self, path):
        img = np.load(path).astype(np.float32)

        # ── Normalise to (C, H, W) ─────────────────────────
        if img.ndim == 2:                        # (H, W)
            img = img[np.newaxis]                # → (1, H, W)
        elif img.ndim == 3:
            if img.shape[0] > 4:                 # (H, W, C)
                img = np.transpose(img, (2, 0, 1))
            # else already (C, H, W)
        return img

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self._load(path)                   # (C, H, W) float32

        # ── Robust per-sample normalisation ─────────────────
        # Clip top/bottom 0.5% to kill hot/cold pixels
        lo, hi = np.percentile(img, 0.5), np.percentile(img, 99.5)
        img = np.clip(img, lo, hi)

        if self.mean is not None and self.std is not None:
            img = (img - self.mean) / (self.std + 1e-8)
        else:
            # Fallback: per-sample z-score so gradients flow
            img = (img - img.mean()) / (img.std() + 1e-8)

        img = torch.from_numpy(img).float()

        # ── Augmentation ────────────────────────────────────
        if self.augment:
            if random.random() > 0.5:
                img = torch.flip(img, dims=[2])
            if random.random() > 0.5:
                img = torch.flip(img, dims=[1])
            k = random.randint(0, 3)
            img = torch.rot90(img, k, dims=[1, 2])

        return img, torch.tensor(label, dtype=torch.long)


In [ ]:
print("\nComputing dataset statistics …")
temp_ds = LensDataset(TRAIN_DIR, mean=None, std=None, augment=False)

pixels = []
for i in range(len(temp_ds)):
    img, _ = temp_ds[i]
    pixels.append(img.numpy())

pixels = np.concatenate(pixels, axis=0).ravel()
MEAN   = float(np.mean(pixels))
STD    = float(np.std(pixels))
print(f"Global mean: {MEAN:.6f}  |  std: {STD:.6f}")


assert STD > 0.01, (
    f"STD={STD:.6f} is suspiciously small — check your .npy files!"
)


In [ ]:
BATCH  = 32
N_WORK = 2

train_ds = LensDataset(TRAIN_DIR, MEAN, STD, augment=True)
val_ds   = LensDataset(VAL_DIR,   MEAN, STD, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=N_WORK, pin_memory=True,
                          drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=N_WORK, pin_memory=True)

imgs, labels = next(iter(train_loader))
print(f"\nBatch  : {imgs.shape}")
print(f"Labels : {torch.unique(labels).tolist()}")
print(f"ImgMin : {imgs.min():.3f}  ImgMax: {imgs.max():.3f}  "
      f"ImgMean: {imgs.mean():.3f}")
print(f"Train  : {len(train_ds)}  |  Val: {len(val_ds)}")





In [ ]:

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
fig.suptitle("Sample Images per Class (after preprocessing)",
             fontsize=13, fontweight='bold')

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    shown = 0
    for path, lbl in train_ds.samples:
        if lbl == cls_idx and shown < 6:
            img = LensDataset(TRAIN_DIR)._load(path)
            img_disp = img[0] if img.shape[0] > 1 else img.squeeze()
            axes[cls_idx, shown].imshow(img_disp, cmap='inferno')
            axes[cls_idx, shown].axis('off')
            if shown == 0:
                axes[cls_idx, 0].set_title(cls_name, fontsize=9,
                                            fontweight='bold')
            shown += 1

plt.tight_layout()
plt.savefig("sample_images.png", dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        prob = F.softmax(out, dim=1)

        total_loss += loss.item()
        preds       = out.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(prob.cpu().numpy())

    return (total_loss / len(loader), correct / total,
            np.array(all_preds), np.array(all_labels),
            np.array(all_probs))


def run_training(model, epochs, lr, wd=1e-4,
                 warmup=3, label='model', patience=12):
    """
    Full training loop with:
      • Linear warm-up + Cosine Annealing LR schedule
      • Mixed Precision (GPU only)
      • Early stopping (patience epochs)
      • Best-model checkpoint
    """
    # CrossEntropy without label smoothing for debugging;
    # class_weight if dataset is imbalanced
    counts = np.bincount([l for _, l in train_ds.samples])
    w      = torch.tensor(1.0 / counts, dtype=torch.float).to(device)
    w      = w / w.sum() * NUM_CLASSES             # normalise
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd,
                            betas=(0.9, 0.999))

    # Warm-up: linear ramp for first `warmup` epochs
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(epochs - warmup, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))   # cosine

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    history      = {k: [] for k in ('train_loss', 'val_loss',
                                     'train_acc',  'val_acc', 'lr')}
    best_val_acc = 0.0
    best_state   = None
    no_improve   = 0

    print(f"\n{'─'*62}")
    print(f"  Training: {label}")
    print(f"  Params  : {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'─'*62}")
    print(f"  {'Ep':>3}  {'T-Loss':>8}  {'T-Acc':>7}  "
          f"{'V-Loss':>8}  {'V-Acc':>7}  {'LR':>10}")
    print(f"{'─'*62}")

    for epoch in range(1, epochs + 1):
        t_loss, t_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler)
        v_loss, v_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['lr'].append(current_lr)

        marker = ' ★' if v_acc > best_val_acc else ''
        print(f"  {epoch:>3}  {t_loss:>8.4f}  {t_acc:>6.2%}  "
              f"{v_loss:>8.4f}  {v_acc:>6.2%}  "
              f"{current_lr:>10.2e}{marker}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve   = 0
        else:
            no_improve  += 1
            if no_improve >= patience:
                print(f"  Early stop at epoch {epoch} "
                      f"(no improvement for {patience} epochs)")
                break

    print(f"{'─'*62}")
    print(f"  Best Val Acc: {best_val_acc:.4%}")
    model.load_state_dict(best_state)
    return history, best_val_acc


def plot_curves(history, title, fname):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], 'b-o', ms=3, label='Train')
    axes[0].plot(epochs, history['val_loss'],   'r-o', ms=3, label='Val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', ms=3, label='Train')
    axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', ms=3, label='Val')
    axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 105); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, history['lr'], 'g-o', ms=3)
    axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
    axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  → saved {fname}")


In [ ]:
class SimpleCNN(nn.Module):
    """
    Fixed baseline CNN.
    Key fixes vs original:
      • BatchNorm after every conv (stabilises training)
      • GELU activations (smoother gradient landscape)
      • Adaptive input conv to handle any IN_CHANNELS
      • AdaptiveAvgPool instead of hard-coded flatten size
        → works for any image resolution
    """
    def __init__(self, in_ch=IN_CHANNELS, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)     # resolution-agnostic
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


In [ ]:
baseline_model = SimpleCNN().to(device)
history_baseline, best_acc_baseline = run_training(
    baseline_model, epochs=20, lr=3e-4, wd=1e-4,
    warmup=5, label="SimpleCNN (Phase 1)", patience=15
)
plot_curves(history_baseline,
            f"Phase 1 — SimpleCNN  (Best Val: {best_acc_baseline:.2%})",
            "phase1_curves.png")

In [ ]:
class SEBlock(nn.Module):
    """
    Channel-wise Squeeze-and-Excitation.
    Learns which feature channels encode Einstein ring brightness,
    arc curvature, and subhalo signatures — suppresses background noise.
    """
    def __init__(self, ch, ratio=8):
        super().__init__()
        mid = max(ch // ratio, 4)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, mid, bias=False), nn.ReLU(),
            nn.Linear(mid, ch, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.net(x).view(x.size(0), x.size(1), 1, 1)

class SpatialAttention(nn.Module):
    """
    CBAM-style spatial attention.
    Focuses computation on the Einstein ring region rather than
    the (uninformative) dark background pixels.
    """
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(1, keepdim=True)
        mx, _ = x.max(1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([avg, mx], 1)))

In [ ]:
class ResBlock(nn.Module):
    """
    Standard residual block: Conv-BN-GELU-Conv-BN + skip connection.
    SE and Spatial attention applied AFTER the residual sum so
    the skip path always carries clean gradient to earlier layers.

    Why GELU over ReLU?
    GELU provides a smoother gradient landscape near zero, which
    benefits training on low-contrast astronomical images where
    many activations are near-zero (black background).
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se      = SEBlock(out_ch)
        self.spatial = SpatialAttention()
        self.act     = nn.GELU()

        # Projection shortcut: only when dimensions change
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        out = self.body(x)
        out = self.act(out + self.skip(x))  # residual sum FIRST
        out = self.se(out)                  # channel attention AFTER
        out = self.spatial(out)             # spatial attention AFTER
        return out

In [ ]:
class ImprovedCNN(nn.Module):
    """
    Corrected ImprovedCNN for gravitational lens classification.

    Architecture design rationale
    ──────────────────────────────
    Stem: 3×3 conv (not 7×7)
      A 7×7 stem was designed for ImageNet (224×224 images). For 150×150
      single-channel astronomical images it over-downsamples too early,
      discarding fine ring structure before any feature is learned.
      A 3×3 stem preserves more spatial detail in early layers.

    4 residual stages (not 5):
      Stage widths: 32 → 64 → 128 → 256
      Each stride-2 stage halves spatial resolution:
        150 → 75 → 37 → 18 → 9
      After 4 stages, spatial resolution is 9×9 — sufficient for GAP
      to produce meaningful channel-wise pooling statistics.
      The 5th stage (→ 512, 3×3→1×1) was killing attention modules.

    Single dropout in head (rate=0.3):
      Moderate dropout only before the final linear layer acts as
      a weak ensemble regulariser without starving the classifier
      of features at training time.

    Layer Norm in head (not Batch Norm):
      BatchNorm1d on the 256-D feature vector depends on batch statistics
      which are noisy at batch_size=32. LayerNorm is instance-wise and
      more stable for small batches.

    Spatial resolution trace (150×150 input):
      Stem (stride 1)         : 150×150 @ 32ch
      Stage 1 (stride 2)      :  75×75  @ 64ch
      Stage 2 (stride 2)      :  37×37  @ 128ch
      Stage 3 (stride 2)      :  18×18  @ 256ch
      Stage 4 (stride 1)      :  18×18  @ 256ch  ← no downsampling
      GAP                     :   1×1   @ 256ch
      Flatten                 :  256-D feature vector
    """
    def __init__(self, in_ch=IN_CHANNELS, num_classes=NUM_CLASSES, drop=0.3):
        super().__init__()

        # ── Stem: gentle entry with 3×3 conv, no stride ──────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )

        # ── Residual stages ───────────────────────────────────────────────
        self.stage1 = ResBlock(32,  64,  stride=2)   # 150→75
        self.stage2 = ResBlock(64,  128, stride=2)   # 75→37
        self.stage3 = ResBlock(128, 256, stride=2)   # 37→18
        self.stage4 = ResBlock(256, 256, stride=1)   # 18→18 (depth, no pooling)

        # ── Global Average Pooling → resolution-invariant descriptor ──────
        self.gap = nn.AdaptiveAvgPool2d(1)            # → (B, 256, 1, 1)

        # ── Classification head ───────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Flatten(),                             # → (B, 256)
            nn.LayerNorm(256),                        # stable for small batches
            nn.Linear(256, 256), nn.GELU(),           # feature mixing
            nn.Dropout(drop),                         # single, moderate dropout
            nn.Linear(256, num_classes)               # output logits
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.gap(x)
        return self.head(x)

    def get_last_conv(self):
        """Return final Conv2d layer — used by Grad-CAM."""
        last = None
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                last = m
        return last

In [ ]:
improved_model = ImprovedCNN().to(device)

total_params = sum(p.numel() for p in improved_model.parameters())
print(f"ImprovedCNN parameters: {total_params:,}")

# Verify spatial resolution trace with a dummy forward pass
with torch.no_grad():
    dummy = torch.zeros(1, IN_CHANNELS, IMG_H, IMG_W).to(device)

    x = improved_model.stem(dummy)
    print(f"After stem    : {tuple(x.shape)}")
    x = improved_model.stage1(x)
    print(f"After stage1  : {tuple(x.shape)}")
    x = improved_model.stage2(x)
    print(f"After stage2  : {tuple(x.shape)}")
    x = improved_model.stage3(x)
    print(f"After stage3  : {tuple(x.shape)}")
    x = improved_model.stage4(x)
    print(f"After stage4  : {tuple(x.shape)}")
    x = improved_model.gap(x)
    print(f"After GAP     : {tuple(x.shape)}")

history_improved, best_acc_improved = run_training(
    improved_model,
    epochs   = 16,     # was 40 — model needs more epochs to converge
    lr       = 3e-4,   # same stable LR as SimpleCNN baseline
    wd       = 1e-4,   # was 1e-5 — slightly stronger weight decay
    warmup   = 5,      # was 0  — prevents BN stat corruption in ep 1-12
    label    = "ImprovedCNN (Phase 2)",
    patience = 20,     # was 15 — cosine schedule can plateau temporarily
)

plot_curves(
    history_improved,
    f"Phase 2 — ImprovedCNN  (Best Val: {best_acc_improved:.2%})",
    "phase2_curves.png"
)

In [ ]:
import torchvision.models as tv_models

def build_resnet18(pretrained=True):
    weights = tv_models.ResNet18_Weights.DEFAULT if pretrained else None
    m = tv_models.resnet18(weights=weights)

    # Adapt first conv for single-channel input
    old = m.conv1
    new = nn.Conv2d(IN_CHANNELS, 64, 7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        if IN_CHANNELS == 1:
            new.weight.copy_(old.weight.mean(1, keepdim=True))
        else:
            new.weight.copy_(old.weight)
    m.conv1 = new

    # Better head: BN → Dropout → FC
    m.fc = nn.Sequential(
        nn.BatchNorm1d(512),
        nn.Dropout(0.4),
        nn.Linear(512, 256), nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(256, NUM_CLASSES)
    )
    return m


resnet18_model = build_resnet18(pretrained=True).to(device)

# ── Phase 3A: head-only warm-up ──────────────────────────────
for name, p in resnet18_model.named_parameters():
    if 'fc' not in name and 'conv1' not in name:
        p.requires_grad = False

history_r18a, _ = run_training(
    resnet18_model, epochs=20, lr=5e-4, wd=1e-4,
    warmup=3, label="ResNet-18 — head warm-up (Phase 3A)", patience=8
)

# ── Phase 3B: full fine-tune with lower LR ───────────────────
for p in resnet18_model.parameters():
    p.requires_grad = True

history_r18b, best_acc_r18 = run_training(
    resnet18_model, epochs=20, lr=5e-5, wd=1e-4,
    warmup=3, label="ResNet-18 — fine-tune (Phase 3B)", patience=15
)

history_r18 = {k: history_r18a[k] + history_r18b[k] for k in history_r18a}
plot_curves(history_r18,
            f"Phase 3 — ResNet-18  (Best Val: {best_acc_r18:.2%})",
            "phase3_curves.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phase 1–3 Model Comparison", fontsize=13, fontweight='bold')

for ax, metric, ylabel in zip(
    axes, ['val_acc', 'val_loss'],
    ['Validation Accuracy (%)', 'Validation Loss']
):
    for hist, name, col in [
        (history_baseline, 'SimpleCNN',   '#4C72B0'),
        (history_improved, 'ImprovedCNN', '#DD8452'),
        (history_r18,      'ResNet-18',   '#55A868'),
    ]:
        vals = [v * 100 for v in hist[metric]] if metric == 'val_acc' else hist[metric]
        ax.plot(vals, label=name, color=col, linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("phase123_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "═"*55)
print("  MODEL COMPARISON")
print("═"*55)
for name, acc in [('SimpleCNN',   best_acc_baseline),
                   ('ImprovedCNN', best_acc_improved),
                   ('ResNet-18',   best_acc_r18)]:
    bar = '█' * int(acc * 40)
    print(f"  {name:<14} {acc:>7.2%}  {bar}")
print("═"*55)

In [ ]:
registry = {
    'SimpleCNN':   (baseline_model, best_acc_baseline),
    'ImprovedCNN': (improved_model, best_acc_improved),
    'ResNet-18':   (resnet18_model, best_acc_r18),
}
best_name  = max(registry, key=lambda k: registry[k][1])
best_model = registry[best_name][0]
print(f"\nBest model: {best_name}  ({registry[best_name][1]:.2%})")

counts   = np.bincount([l for _, l in train_ds.samples])
w        = torch.tensor(1.0 / counts, dtype=torch.float).to(device)
w        = w / w.sum() * NUM_CLASSES
crit_eval = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

_, v_acc, preds, labels_true, probs = evaluate(
    best_model, val_loader, crit_eval)
print(f"Final val accuracy : {v_acc:.4%}")

In [ ]:
cm = confusion_matrix(labels_true, preds)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("phase4_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n" + classification_report(labels_true, preds,
                                    target_names=CLASS_NAMES, digits=4))


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def get_roc_data(model):
    model.eval()

    _, _, preds, labels, probs = evaluate(model, val_loader, crit_eval)

    labels_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(NUM_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(labels_bin.ravel(), probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    return fpr, tpr, roc_auc

In [ ]:
fpr_base, tpr_base, auc_base = get_roc_data(baseline_model)
fpr_imp,  tpr_imp,  auc_imp  = get_roc_data(improved_model)
fpr_res,  tpr_res,  auc_res  = get_roc_data(resnet18_model)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(fpr_base["micro"], tpr_base["micro"],
         label=f"SimpleCNN (AUC={auc_base['micro']:.3f})")

plt.plot(fpr_imp["micro"], tpr_imp["micro"],
         label=f"ImprovedCNN (AUC={auc_imp['micro']:.3f})")

plt.plot(fpr_res["micro"], tpr_res["micro"],
         label=f"ResNet18 (AUC={auc_res['micro']:.3f})")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(alpha=0.3)

plt.show()